# 04 CatBoost Baseline
Этапы 9-10: baseline CatBoost training/inference с сохранением артефактов.

In [1]:
import importlib
import json
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from _shared_notebook_utils import RESEARCH_CHECKPOINT_DIR, CATBOOST_DIR, save_json

# Load frozen split from checkpoint built in notebook 03.
baseline_dir = RESEARCH_CHECKPOINT_DIR / "baseline"
if not baseline_dir.exists():
    raise FileNotFoundError(f"Missing baseline checkpoint dir: {baseline_dir}")

train_df = pd.read_pickle(baseline_dir / "train_df.pkl")
val_df = pd.read_pickle(baseline_dir / "val_df.pkl")
test_df = pd.read_pickle(baseline_dir / "test_df.pkl")

baseline_meta_path = RESEARCH_CHECKPOINT_DIR / "baseline_meta.json"
if baseline_meta_path.exists():
    baseline_meta = json.loads(baseline_meta_path.read_text(encoding="utf-8"))
    target_to_id = {str(k): int(v) for k, v in baseline_meta["target_to_id"].items()}
    id_to_target = {int(k): str(v) for k, v in baseline_meta["id_to_target"].items()}
else:
    classes = sorted(train_df["target"].astype("string").dropna().unique().tolist())
    target_to_id = {name: idx for idx, name in enumerate(classes)}
    id_to_target = {idx: name for name, idx in target_to_id.items()}

# Feature schema: prefer existing baseline model meta if present, fallback to expected columns.
model_meta_path = CATBOOST_DIR / "baseline" / "meta.json"
if model_meta_path.exists():
    model_meta_existing = json.loads(model_meta_path.read_text(encoding="utf-8"))
    feature_columns = list(model_meta_existing.get("feature_columns", []))
    categorical_features = list(model_meta_existing.get("categorical_features", []))
    numeric_features = list(model_meta_existing.get("numeric_features", []))
else:
    feature_columns = [
        c for c in [
            "history_1", "history_2", "history_3",
            "STATEFIPS", "CNTYFIPS",
            "CSBACRES", "INSIDE_X", "INSIDE_Y",
        ] if c in train_df.columns
    ]
    categorical_features = [c for c in ["history_1", "history_2", "history_3", "STATEFIPS", "CNTYFIPS"] if c in feature_columns]
    numeric_features = [c for c in ["CSBACRES", "INSIDE_X", "INSIDE_Y"] if c in feature_columns]

for split_df in [train_df, val_df, test_df]:
    if "target_encoded" not in split_df.columns:
        split_df["target_encoded"] = split_df["target"].astype("string").map(target_to_id).astype(int)
    for col in categorical_features:
        split_df[col] = split_df[col].astype("string")

missing_features = [c for c in feature_columns if c not in train_df.columns]
if missing_features:
    raise ValueError(f"Missing training features: {missing_features}")

X_train = train_df[feature_columns]
X_val = val_df[feature_columns]
X_test = test_df[feature_columns]

y_train = train_df["target_encoded"].astype(int)
y_val = val_df["target_encoded"].astype(int)
y_test = test_df["target_encoded"].astype(int)

labels_sorted = sorted(id_to_target.keys())
target_names = [id_to_target[i] for i in labels_sorted]

print("Loaded split dataframes:", train_df.shape, val_df.shape, test_df.shape)
print("feature_columns:", feature_columns)
print("categorical_features:", categorical_features)
print("n_classes:", len(target_names))

Loaded split dataframes: (26957630, 12) (5776566, 12) (5777014, 12)
feature_columns: ['history_1', 'history_2', 'history_3', 'STATEFIPS', 'CNTYFIPS', 'CSBACRES', 'INSIDE_X', 'INSIDE_Y']
categorical_features: ['history_1', 'history_2', 'history_3', 'STATEFIPS', 'CNTYFIPS']
n_classes: 9


In [2]:
# 4.1 Train/evaluate/save baseline CatBoost
RUN_TRAIN = False
RUN_EVAL = True
RUN_SAVE_MODEL = False
USE_GPU = True

baseline_artifacts_dir = CATBOOST_DIR / "baseline"
baseline_artifacts_dir.mkdir(parents=True, exist_ok=True)
model_path = baseline_artifacts_dir / "model.cbm"
meta_path = baseline_artifacts_dir / "meta.json"
baseline_train_dir = baseline_artifacts_dir / "catboost_info"
baseline_train_dir.mkdir(parents=True, exist_ok=True)

cat_feature_indices = [feature_columns.index(col) for col in categorical_features]

catboost_mod = importlib.import_module("catboost")
CatBoostClassifier = catboost_mod.CatBoostClassifier

if RUN_TRAIN:
    catboost_params = {
        "loss_function": "MultiClass",
        "eval_metric": "TotalF1:average=Macro",
        "iterations": 1000,
        "learning_rate": 0.05,
        "depth": 6,
        "l2_leaf_reg": 3.0,
        "random_seed": 42,
        "early_stopping_rounds": 50,
        "task_type": "GPU" if USE_GPU else "CPU",
        "train_dir": baseline_train_dir.as_posix(),
        "verbose": 100,
    }
    if USE_GPU:
        catboost_params["devices"] = "0"

    cat_model = CatBoostClassifier(**catboost_params)
    cat_model.fit(
        X_train,
        y_train,
        cat_features=cat_feature_indices,
        eval_set=(X_val, y_val),
        use_best_model=True,
    )

    print("Training completed.")
    print("Best iteration:", cat_model.get_best_iteration())
    print("Best validation score:", cat_model.get_best_score())
elif model_path.exists():
    cat_model = CatBoostClassifier()
    cat_model.load_model(model_path.as_posix())
    print("Loaded existing baseline model:", model_path.resolve())
else:
    raise ValueError("No model is available. Set RUN_TRAIN=True or provide artifacts/catboost/baseline/model.cbm")

def flatten_catboost_preds(pred):
    return np.asarray(pred).reshape(-1).astype(int)

if RUN_EVAL:
    val_pred = flatten_catboost_preds(cat_model.predict(X_val))
    test_pred = flatten_catboost_preds(cat_model.predict(X_test))

    val_acc = accuracy_score(y_val, val_pred)
    test_acc = accuracy_score(y_test, test_pred)
    val_f1_macro = f1_score(y_val, val_pred, average="macro", zero_division=0)
    test_f1_macro = f1_score(y_test, test_pred, average="macro", zero_division=0)
    val_f1_weighted = f1_score(y_val, val_pred, average="weighted", zero_division=0)
    test_f1_weighted = f1_score(y_test, test_pred, average="weighted", zero_division=0)

    print("Validation metrics:")
    print("accuracy:", f"{float(val_acc):.6f}")
    print("macro F1:", f"{float(val_f1_macro):.6f}")
    print("weighted F1:", f"{float(val_f1_weighted):.6f}")

    print("\nTest metrics:")
    print("accuracy:", f"{float(test_acc):.6f}")
    print("macro F1:", f"{float(test_f1_macro):.6f}")
    print("weighted F1:", f"{float(test_f1_weighted):.6f}")

    print("\nClassification report (validation):")
    print(classification_report(y_val, val_pred, labels=labels_sorted, target_names=target_names, zero_division=0))

    print("Classification report (test):")
    print(classification_report(y_test, test_pred, labels=labels_sorted, target_names=target_names, zero_division=0))

    cm_test_df = pd.DataFrame(
        confusion_matrix(y_test, test_pred, labels=labels_sorted),
        index=[f"true_{id_to_target[i]}" for i in labels_sorted],
        columns=[f"pred_{id_to_target[i]}" for i in labels_sorted],
    )
    print("Confusion matrix (test):")
    display(cm_test_df)

if RUN_SAVE_MODEL:
    cat_model.save_model(model_path.as_posix())
    model_meta = {
        "feature_columns": feature_columns,
        "categorical_features": categorical_features,
        "numeric_features": numeric_features,
        "target_to_id": target_to_id,
        "id_to_target": {str(k): v for k, v in id_to_target.items()},
    }
    save_json(model_meta, meta_path)
    print("Model saved to:", model_path.resolve())
    print("Metadata saved to:", meta_path.resolve())

Loaded existing baseline model: C:\Users\Dmitry\code-projects\diploma-crop-rotation\artifacts\catboost\baseline\model.cbm
Validation metrics:
accuracy: 0.699281
macro F1: 0.568771
weighted F1: 0.689966

Test metrics:
accuracy: 0.699006
macro F1: 0.568485
weighted F1: 0.689626

Classification report (validation):
               precision    recall  f1-score   support

         corn       0.69      0.72      0.71   1763620
       cotton       0.72      0.77      0.74    278319
       fallow       0.59      0.41      0.48    270527
   forage_hay       0.81      0.85      0.83    561759
      legumes       0.50      0.14      0.22     86921
other_cereals       0.53      0.24      0.33    150744
      sorghum       0.50      0.31      0.38    173768
     soybeans       0.72      0.76      0.74   1645760
        wheat       0.66      0.73      0.69    845148

     accuracy                           0.70   5776566
    macro avg       0.64      0.55      0.57   5776566
 weighted avg       0.69

,pred_corn,pred_cotton,pred_fallow,pred_forage_hay,pred_legumes,pred_other_cereals,pred_sorghum,pred_soybeans,pred_wheat
true_corn,1266553,26448,17973,58630,2484,5805,12360,317454,54675
true_cotton,16234,212336,2279,1598,26,388,6933,18193,19236
true_fallow,34538,5407,110117,4241,894,4437,11882,45449,53721
true_forage_hay,43086,1643,2255,476608,476,5738,703,14132,15751
true_legumes,12619,288,1886,2492,12137,1955,203,16137,39947
true_other_cereals,29498,2425,9440,12649,537,36335,2906,11092,46478
true_sorghum,29854,13842,18096,2328,47,2203,53133,9355,44719
true_soybeans,307096,14165,4838,12571,3231,898,3237,1251502,48083
true_wheat,88648,17502,20754,17727,3983,11039,16629,52381,619446


### 4.2 Inference-only path from saved artifacts
Эта ячейка используется для быстрого инференса без переобучения модели.

In [ ]:
# 4.2 Load saved CatBoost model and run inference
baseline_artifacts_dir = CATBOOST_DIR / "baseline"
model_path = baseline_artifacts_dir / "model.cbm"
meta_path = baseline_artifacts_dir / "meta.json"

if not model_path.exists() or not meta_path.exists():
    raise FileNotFoundError(
        f"Saved artifacts were not found. Expected files: {model_path} and {meta_path}"
    )

catboost_mod = importlib.import_module("catboost")
CatBoostClassifier = catboost_mod.CatBoostClassifier

loaded_cat_model = CatBoostClassifier()
loaded_cat_model.load_model(model_path.as_posix())
print("Loaded model from:", model_path.resolve())

with open(meta_path, "r", encoding="utf-8") as f:
    loaded_meta = json.load(f)
print("Loaded metadata keys:", list(loaded_meta.keys()))

loaded_feature_columns = loaded_meta["feature_columns"]
id_to_target_loaded = {int(k): v for k, v in loaded_meta["id_to_target"].items()}

X_infer = X_test[loaded_feature_columns]
pred_ids = np.asarray(loaded_cat_model.predict(X_infer)).reshape(-1).astype(int)
pred_labels = pd.Series(pred_ids, index=X_infer.index).map(id_to_target_loaded)

prediction_preview = pd.DataFrame({"pred_id": pred_ids, "pred_label": pred_labels})
print("Rows predicted:", len(X_infer))
display(prediction_preview.head(20))

loaded_test_acc = accuracy_score(y_test, pred_ids)
print("Loaded model test accuracy:", f"{float(loaded_test_acc):.6f}")

Loaded model from: C:\Users\Dmitry\code-projects\diploma-crop-rotation\artifacts\catboost\baseline\model.cbm
Loaded metadata keys: ['feature_columns', 'categorical_features', 'numeric_features', 'target_to_id', 'id_to_target']
Rows predicted: 5777014


,pred_id,pred_label
2,3,forage_hay
11,3,forage_hay
15,6,sorghum
22,8,wheat
29,6,sorghum
36,3,forage_hay
42,3,forage_hay
52,6,sorghum
53,6,sorghum
65,8,wheat


Loaded model test accuracy: 0.699006


: 